<a href="https://colab.research.google.com/github/rishikreddyv/gemini-rag-agent-project/blob/main/gemini_rag_agent_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q langchain langchain-community langchain-google-genai faiss-cpu pypdf sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 21.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth

In [ ]:
import os
os.environ["GOOGLE_API_KEY"] = "YOUR_API_KEY_HERE"

In [ ]:
from google.colab import files
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]

Saving DeltaX.pdf to DeltaX.pdf


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader(pdf_path)
docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(docs)
print(f"{len(chunks)} chunks created")

5 chunks created


In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

/tmp/ipykernel_4481/316697040.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (4 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash", temperature=0)

In [ ]:
import os
key = os.environ.get("GOOGLE_API_KEY", "NOT SET")
print(f"Key starts with: {key[:8]}..." if key != "NOT SET" else "NOT SET")

Key starts with: AQ.Ab8RN...


In [ ]:
query = "Summarize this document in one sentence"
relevant_docs = retriever.invoke(query)
context = "\n\n".join([doc.page_content for doc in relevant_docs])
prompt = f"""Answer the question based only on the context below.
Context:
{context}
Question: {query}
Answer:"""
response = llm.invoke(prompt)
print(response.content)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'This document presents the resume of Vendidandi Rishik Reddy, a Computer Science undergraduate at K L University specializing in AI-Driven Language Technologies, highlighting his skills and projects in full-stack web development and natural language processing.', 'extras': {'signature': 'EqMPCqAPARFNMg+5jZvhDbWvXcRiEYfjnaLUAGZJzztPLGG3RdL4S34a/T7CLHMst7lj5D0hFi64p8w23dYKuIprfYWqWlnWXbibuG6oufk5jCzowCXVevzUGcJsuMYm2qVaiR4He2l/+02YroLP42hsFk/R+UMlaR9PptAcd90awK2YUBxNinsl5EOUstyI/H8gDoWVKhcxG+SEK6EXRF+bbZjN+HqVrCAF6LPASaa/+9qLuMDjKfCPkqSlTOsgilRf6K79sgOQ2AerKKwbu9+XOcIWU2OBeyZVRL/YulaFa9BrUO1QI5pb69Gn3vmer3krcHYBw9h1TFTrqQyb32MvLGdkJ+SQXdLBDw4hG4kdjRXlsM0aCNqPHG/ZcO4LJtbmvnxYfstgPQMJdgh6ljWg6dhW/1Rpoz0Tp8JZlXzEHr9QojiNDU8yAeNXZ0AnGIwuY4AYhpNE0N03V3fZRVqcl5scrtrlv+yYEnhvNjrp7ravjFjEj7faVqIQGHhK6NXFCLEVfpcDQftXPCB1wIJ3FlLat7SQgJyrYECjtRM+o3ylz/rlYPXoTxn/I6bRpx3ZhHRbVOIM/l3B5jop+mgbpuTilMO9FsmlAOhrkE+PeKk+7uzhLqr+oMk7osIQP+l+CUqur+pO8TvtJQitDXMAQUOg4KaebqNdtzyqK1pN

In [ ]:
import os
key = os.environ.get("GOOGLE_API_KEY", "NOT SET")
print(f"Length: {len(key)}")
print(f"First 6 chars: {key[:6]}")
print(f"Last 4 chars: {key[-4:]}")

Length: 53
First 6 chars: AQ.Ab8
Last 4 chars: QsjQ


In [ ]:
response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents="Say hello in one word"
)
print(response.text)

Hello


In [ ]:
!pip install -q -U ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 78.0 MB/s eta 0:00:00


In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun
from langchain.tools import tool

search = DuckDuckGoSearchRun()

@tool
def calculator(expression: str) -> str:
    """Use this for math. Input must be a plain math expression like '25*4'."""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"

@tool
def web_search(query: str) -> str:
    """Use this to answer questions about current events or facts you don't know."""
    return search.run(query)

tools = [calculator, web_search]

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.prebuilt import create_react_agent

agent_llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash", temperature=0)
agent = create_react_agent(agent_llm, tools)

/tmp/ipykernel_4481/2245278577.py:5: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(agent_llm, tools)


In [ ]:
response = agent.invoke({"messages": [("user", "What is 47 * 23, and who is the current CEO of OpenAI?")]})

for message in response["messages"]:
    print(f"{message.type}: {message.content}")

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


human: What is 47 * 23, and who is the current CEO of OpenAI?
ai: []
tool: 1081
tool: 10 hours ago - Altman co-founded OpenAI in 2015 and became its CEO in 2019, a role that made him a prominent figure of the AI boom. He supervised the launch of ChatGPT in November 2022. In 2023, he was ousted by the organization's board of directors for not being "consistently candid". 12 hours ago - In June 2026 it filed for an initial public offering. OpenAI has faced multiple lawsuits for alleged deaths linked to its chatbots and copyright infringement against authors and media companies. In November 2023, OpenAI's board removed Sam Altman as CEO, citing a lack of confidence, but reinstated ... 1 week ago - Sam Altman says OpenAI has fumbled at communicating AI benefits • Sep. 4, 2026, 10:40 AM ET(The San Jose Mercury News)...(Show more) ... Sam Altman was president of the start-up accelerator Y Combinator from 2014 to 2019 and became CEO of ... July 22, 2026 - OpenAI CEO Sam Altman has become one 

In [1]:
!mkdir -p src

In [2]:
!pip install -q langchain-huggingface rank-bm25

In [3]:
%%writefile src/config.py
import os
from dataclasses import dataclass


@dataclass
class Settings:
    google_api_key: str = os.getenv("GOOGLE_API_KEY", "")
    gemini_model: str = "gemini-3.6-flash"
    embedding_model: str = "all-MiniLM-L6-v2"
    reranker_model: str = "cross-encoder/ms-marco-MiniLM-L-6-v2"
    chunk_size: int = 600
    chunk_overlap: int = 100
    initial_k: int = 6
    top_k: int = 3
    faiss_dir: str = "faiss_index"


settings = Settings()

Writing src/config.py


In [4]:
%%writefile src/ingest.py
import os
import pickle
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

from src.config import settings


def ingest_pdf(pdf_path: str, output_dir: str = settings.faiss_dir) -> int:
    loader = PyPDFLoader(pdf_path)
    docs = loader.load()

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=settings.chunk_size,
        chunk_overlap=settings.chunk_overlap,
    )
    chunks = splitter.split_documents(docs)

    for chunk in chunks:
        raw_page = chunk.metadata.get("page", 0)
        chunk.metadata["page_number"] = raw_page + 1 if isinstance(raw_page, int) else raw_page
        chunk.metadata["source_file"] = os.path.basename(pdf_path)

    embeddings = HuggingFaceEmbeddings(model_name=settings.embedding_model)
    vectorstore = FAISS.from_documents(chunks, embeddings)

    os.makedirs(output_dir, exist_ok=True)
    vectorstore.save_local(output_dir)

    with open(os.path.join(output_dir, "chunks.pkl"), "wb") as f:
        pickle.dump(chunks, f)

    print(f"Saved {len(chunks)} chunks from '{pdf_path}' into '{output_dir}/'")
    return len(chunks)

Writing src/ingest.py


In [6]:
!pip install -q langchain langchain-community langchain-text-splitters langchain-google-genai langchain-huggingface faiss-cpu rank-bm25 pypdf sentence-transformers

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.63.1 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.6.1 which is incompatible.


In [7]:
from google.colab import files
from src.ingest import ingest_pdf

uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]
ingest_pdf(pdf_path)

Saving Resume.pdf to Resume.pdf


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (6 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Saved 5 chunks from 'Resume.pdf' into 'faiss_index/'


5

In [8]:
%%writefile src/retriever.py
import os
import pickle
from typing import List
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from sentence_transformers import CrossEncoder

from src.config import settings


class HybridRerankRetriever:
    def __init__(self, index_dir: str = settings.faiss_dir):
        if not os.path.exists(index_dir):
            raise FileNotFoundError(f"Index directory '{index_dir}' not found.")

        embeddings = HuggingFaceEmbeddings(model_name=settings.embedding_model)
        self.vectorstore = FAISS.load_local(
            index_dir, embeddings, allow_dangerous_deserialization=True
        )

        chunks_path = os.path.join(index_dir, "chunks.pkl")
        with open(chunks_path, "rb") as f:
            self.chunks: List[Document] = pickle.load(f)

        k_val = min(settings.initial_k, len(self.chunks)) if self.chunks else 1
        self.bm25 = BM25Retriever.from_documents(self.chunks)
        self.bm25.k = k_val
        self.reranker = CrossEncoder(settings.reranker_model)

    def retrieve(self, query: str, top_k: int = settings.top_k) -> List[Document]:
        if not self.chunks:
            return []

        k_val = min(settings.initial_k, len(self.chunks))
        vector_docs = self.vectorstore.similarity_search(query, k=k_val)
        bm25_docs = self.bm25.invoke(query)

        seen = set()
        candidates: List[Document] = []
        for doc in vector_docs + bm25_docs:
            key = (doc.page_content, doc.metadata.get("page_number"))
            if key not in seen:
                seen.add(key)
                candidates.append(doc)

        pairs = [(query, doc.page_content) for doc in candidates]
        scores = self.reranker.predict(pairs)

        ranked = sorted(zip(candidates, scores), key=lambda item: item[1], reverse=True)
        return [doc for doc, _ in ranked[:top_k]]

    def format_with_citations(self, docs: List[Document]) -> str:
        if not docs:
            return "No relevant context found in the indexed documents."

        blocks = []
        for doc in docs:
            source = doc.metadata.get("source_file", "document")
            page = doc.metadata.get("page_number", "?")
            blocks.append(f"[{source} | Page {page}]\n{doc.page_content}")
        return "\n\n".join(blocks)

Writing src/retriever.py


In [9]:
from src.retriever import HybridRerankRetriever

retriever = HybridRerankRetriever()
docs = retriever.retrieve("What projects or skills are listed?")
print(retriever.format_with_citations(docs))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

[Resume.pdf | Page 1]
Specialization: AI-Driven Language Technologies
Relevant Coursework: Data Structures & Algorithms, Object-Oriented Programming, Artificial Intel-
ligence & Machine Learning, Deep Learning for Natural Language Processing, Large Language Model for
Speech Processing, Conversational AI and Chatbot Development
Skills
Languages: Python, Java, SQL, JavaScript
AI/ML: Scikit-learn, PyTorch, Hugging Face Transformers, Pandas, NLP
GenAI: LLMs, LLM APIs, LangChain, RAG, Prompt Engineering
Frameworks: Spring Boot, React.js, Streamlit
Tools & Technologies: MySQL, Git, REST APIs, CLI
Projects

[Resume.pdf | Page 1]
Vendidandi Rishik Reddy
/github rishikreddyv | /linkedin rishikreddyv | ✉ rishikreddyv@gmail.com | ♂phone +91 95338 58585
Summary
Computer Science undergraduate specializing in AI-Driven Language Technologies, with a strong foun-
dation in DSA, OOP, and software development. Skilled in Python, Java, and SQL, with hands-on
experience in NLP, machine learning, and full-

In [10]:
%%writefile src/agent.py
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_google_genai import ChatGoogleGenerativeAI

from src.config import settings
from src.retriever import HybridRerankRetriever

_retriever = HybridRerankRetriever()
_ddg = DuckDuckGoSearchRun()


@tool
def search_documents(query: str) -> str:
    """Search the uploaded PDF documents for relevant facts, skills, projects, or details. Always include page and file citations in your answer."""
    docs = _retriever.retrieve(query)
    return _retriever.format_with_citations(docs)


@tool
def calculator(expression: str) -> str:
    """Use this for math. Input must be a plain math expression like '25*4'."""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"


@tool
def web_search(query: str) -> str:
    """Use this to answer questions about current events or facts not found in the uploaded documents."""
    return _ddg.run(query)


def build_agent():
    llm = ChatGoogleGenerativeAI(model=settings.gemini_model, temperature=0)
    tools = [search_documents, calculator, web_search]
    system_prompt = (
        "You are a helpful AI research assistant. "
        "When answering from uploaded documents, cite the exact [file | Page X] source from the tool output. "
        "If the answer is not in the documents and not a math/current-event query, say clearly that it is not found in the provided document rather than guessing."
    )
    return create_agent(model=llm, tools=tools, system_prompt=system_prompt)

Writing src/agent.py
